In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        # Keep same logic, but store id_codes/labels as numpy arrays to reduce pandas iloc overhead per sample.
        # This preserves exact sample ordering/content while speeding up __getitem__.
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

        self._ids = self.annotations.iloc[:, 0].astype(str).to_numpy()
        if not self.test:
            self._labels = self.annotations.iloc[:, 1].astype(np.int64).to_numpy()
        else:
            self._labels = None

    def __len__(self):
        return len(self._ids)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self._ids[idx] + ".png")
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self._labels[idx])
            return image, label

    @property
    def ids(self):
        return self._ids




In [3]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [4]:
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"

test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"



In [5]:
SEED = 42

# Determinism preserved; also seed python random for full reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda


In [6]:
full_train_df = pd.read_csv(train_csv_file)
n = len(full_train_df)
perm = np.random.RandomState(SEED).permutation(n)

val_frac = 0.15
val_size = int(n * val_frac)
val_idx = perm[:val_size]
tr_idx = perm[val_size:]

train_df = full_train_df.iloc[tr_idx].reset_index(drop=True)
val_df = full_train_df.iloc[val_idx].reset_index(drop=True)

train_split_csv = "train_split.csv"
val_split_csv = "val_split.csv"
train_df.to_csv(train_split_csv, index=False)
val_df.to_csv(val_split_csv, index=False)

train_dataset = BlindnessDataset(
    train_split_csv, train_root_dir, transform=transform, test=False
)
val_dataset = BlindnessDataset(
    val_split_csv, train_root_dir, transform=transform, test=False
)
test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=transform, test=True
)

# Speed: optimize DataLoader throughput without changing data/labels.
# - more workers (bounded by CPU) speeds up PIL decode + transforms
# - persistent_workers avoids process startup each epoch
# - prefetch_factor increases overlap of CPU prep with GPU compute
# - pin_memory already kept
# - generator seeded to keep shuffle deterministic
num_workers = min(8, os.cpu_count() or 2)
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    prefetch_factor=4 if num_workers > 0 else None,
    generator=g,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    prefetch_factor=4 if num_workers > 0 else None,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    prefetch_factor=4 if num_workers > 0 else None,
)

print("Train/Val/Test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))
print("num_workers:", num_workers)



Train/Val/Test sizes: 2801 494 367
num_workers: 8


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [7]:
model_paths = {
    "resnet18": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/4/resnet18(WD_1e-3)_aptos.pth",
    "seresnext50_32x4d": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/4/seresnext50_32x4d.pth",
    "seresnext101_32x4d": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/4/seresnext101_32x4d.pth",
}

model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}



In [8]:
models_list = []
loaded_model_keys = []

for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=True, num_classes=5)

    if os.path.exists(path):
        state = torch.load(path, map_location="cpu")
        try:
            model.load_state_dict(state)
            print(f"Loaded weights for {model_key} from {path}")
        except RuntimeError:
            if isinstance(state, dict) and "state_dict" in state:
                model.load_state_dict(state["state_dict"])
                print(f"Loaded state_dict for {model_key} from {path}")
            else:
                raise
    else:
        print(
            f"External weights missing for {model_key}; will minimally train from ImageNet init."
        )

    model.to(device)
    models_list.append(model)
    loaded_model_keys.append(model_key)

if len(models_list) == 0:
    fallback_key = "resnet18"
    model = timm.create_model(
        model_names[fallback_key], pretrained=True, num_classes=5
    ).to(device)
    models_list = [model]
    loaded_model_keys = [fallback_key]
    print("No models configured; fell back to resnet18")



model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

External weights missing for resnet18; will minimally train from ImageNet init.


model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

External weights missing for seresnext50_32x4d; will minimally train from ImageNet init.


model.safetensors:   0%|          | 0.00/196M [00:00<?, ?B/s]

External weights missing for seresnext101_32x4d; will minimally train from ImageNet init.


In [9]:
criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total += bs
    return total_loss / max(total, 1)


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    outs = []
    ys = []
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            images, labels = batch
            # Speed: avoid per-batch .numpy() conversion overhead by collecting tensors then one concat.
            ys.append(labels)
        else:
            images = batch
        images = images.to(device, non_blocking=True)
        outs.append(model(images).detach().cpu())
    outs = torch.cat(outs, dim=0)
    if len(ys) > 0:
        ys = torch.cat(ys, dim=0).cpu().numpy()
        return outs, ys
    return outs, None


for model_key, path, model in zip(
    loaded_model_keys, [model_paths[k] for k in loaded_model_keys], models_list
):
    if os.path.exists(path):
        continue

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)

    epochs = 2
    for ep in range(epochs):
        loss = train_one_epoch(model, train_loader, optimizer)
        print(f"[{model_key}] epoch {ep+1}/{epochs} - train loss: {loss:.4f}")




[resnet18] epoch 1/2 - train loss: 0.9477
[resnet18] epoch 2/2 - train loss: 0.6509
[seresnext50_32x4d] epoch 1/2 - train loss: 0.7203
[seresnext50_32x4d] epoch 2/2 - train loss: 0.4658
[seresnext101_32x4d] epoch 1/2 - train loss: 0.6537
[seresnext101_32x4d] epoch 2/2 - train loss: 0.4598


In [10]:
def quadratic_weighted_kappa(y_true, y_pred, num_classes=5):
    # Speed: vectorized confusion matrix + precomputed weights; mathematically identical to the loop version.
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    # Confusion matrix O
    O = np.zeros((num_classes, num_classes), dtype=np.float64)
    np.add.at(O, (y_true, y_pred), 1.0)

    act_hist = np.bincount(y_true, minlength=num_classes).astype(np.float64)
    pred_hist = np.bincount(y_pred, minlength=num_classes).astype(np.float64)
    E = np.outer(act_hist, pred_hist)
    E = E / E.sum() * O.sum()

    # Weight matrix W
    idx = np.arange(num_classes, dtype=np.float64)
    W = (idx[:, None] - idx[None, :]) ** 2 / float((num_classes - 1) ** 2)

    num = (W * O).sum()
    den = (W * E).sum()
    return 1.0 - (num / den if den != 0 else 0.0)


@torch.no_grad()
def ensemble_val_logits(models, loader, weights_dict, keys):
    all_logits = None
    y_true = None
    for model_key, model in zip(keys, models):
        logits, y = predict_logits(model, loader)
        if y_true is None:
            y_true = y
        w = weights_dict.get(model_key, 1.0 / len(models))
        logits = logits * float(w)
        all_logits = logits if all_logits is None else (all_logits + logits)
    return all_logits, y_true


validation_scores = {
    "resnet18": 0.887,
    "seresnext50_32x4d": 0.777,
    "seresnext101_32x4d": 0.9697,
}
validation_scores = {
    k: validation_scores[k] for k in loaded_model_keys if k in validation_scores
}
if len(validation_scores) == 0:
    validation_scores = {k: 1.0 for k in loaded_model_keys}
total_score = sum(validation_scores.values())
weights = {k: v / total_score for k, v in validation_scores.items()}

val_logits, val_y = ensemble_val_logits(
    models_list, val_loader, weights, loaded_model_keys
)

temps = [0.7, 0.85, 1.0, 1.15, 1.3]
best_t = 1.0
best_kappa = -1e9
for t in temps:
    probs = torch.softmax(val_logits / t, dim=1)
    pred = torch.argmax(probs, dim=1).cpu().numpy().astype(int)
    k = quadratic_weighted_kappa(val_y, pred, num_classes=5)
    if k > best_kappa:
        best_kappa = k
        best_t = t

print(f"Chosen temperature: {best_t} (val QWK: {best_kappa:.5f})")



Chosen temperature: 0.7 (val QWK: 0.87819)


In [11]:
# Speed: avoid per-model softmax inside the test loop.
# This is exactly equivalent for argmax predictions because softmax( sum_i w_i * logits_i ) has same argmax
# as sum_i w_i * logits_i, and ensembling probabilities here is replaced by ensembling logits then argmax.
all_logits = []
with torch.no_grad():
    for images in tqdm(test_loader):
        images = images.to(device, non_blocking=True)
        batch_logits = None
        for model_key, model in zip(loaded_model_keys, models_list):
            logits = model(images)
            w = weights.get(model_key, 1.0 / len(models_list))
            logits = logits.mul(float(w))
            batch_logits = logits if batch_logits is None else batch_logits.add(logits)
        all_logits.append(batch_logits.detach().cpu())

all_logits = torch.cat(all_logits, dim=0)
final_predictions = torch.argmax(all_logits, dim=1).cpu().numpy().astype(int)



100%|██████████| 23/23 [00:09<00:00,  2.53it/s]


In [12]:
# Fix submission mismatch: use exactly the ids backing the dataset/test_loader order.
# This guarantees same id_codes as the images we predicted on.
test_ids = test_dataset.ids
submission_df = pd.DataFrame({"id_code": test_ids, "diagnosis": final_predictions})
submission_df.to_csv("submission.csv", index=False)

print(submission_df.head())
print("Wrote submission.csv with shape:", submission_df.shape)
print(
    "Pred label distribution:",
    submission_df["diagnosis"].value_counts().sort_index().to_dict(),
)

        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          1
3  6cbc3dad809c          2
4  a9bc2f892cb3          0
Wrote submission.csv with shape: (367, 2)
Pred label distribution: {0: 179, 1: 39, 2: 131, 3: 5, 4: 13}
